# ChaoSSM — Phase 2b: Robustness Phase

## Why this phase

The last item in §9.1. The Q1-venue validation of this proposal flagged its absence as
blocking: §4.4 deliberately operates at positive λ — sensitive dependence on initial
conditions — without testing what that costs, despite Halloran et al. (2024) framing Mamba's
λ ≤ 0 as valuable precisely because it delivers stability under mixed precision and
fine-tuning. §5.2/§5.4 gave a partial, indirect answer via training-time decay from peak
accuracy. This phase tests the three costs directly, post-hoc, on frozen trained models.

## Scope

Three tests, on the two tasks that gave the pilot's clearest, opposite-direction results —
**copy** (chaos wins) and **associative recall** (chaos loses, now with a demonstrated
mechanism from Phase 2a) — so any robustness cost can be read against a task where chaos
helps and one where it already hurts, rather than assessed in isolation:

- **Input perturbation**: accuracy under Gaussian noise added to the embedded input at
  increasing magnitude.
- **Quantization sensitivity**: accuracy under weight quantization at decreasing bit-width,
  and under fp16 vs. fp32 inference.
- **Fine-tuning stability**: accuracy drop immediately after a short burst of aggressive
  (10x learning rate) further training — a direct, if small-scale, test of Halloran et al.'s
  framing.

**Scoping decision, stated explicitly**: v2.1's §9.1 also proposed comparing against
Mamba-2/Mamba-3 "where feasible." That is not feasible at this pilot's scale and budget —
implementing and fairly tuning real Mamba baselines is a substantial undertaking in its own
right, better placed in the §9.2 scaling phase where it can be done properly. This phase
compares only against the matched linear-SSM baseline used throughout §5, which is the fair,
apples-to-apples comparison available at this scale.

3 arms x 2 tasks x 6 seeds = 36 base models (reduced from §5's usual protocol to keep this
phase single-GPU and low-cost, per its stated scope); each robustness test is then a forward-
or short-training-pass evaluation on the frozen trained model, not a new training run.

## Setup

In [1]:
import os, json, math, time, warnings, copy as pycopy
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import wilcoxon
from torch.func import jacrev

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)

def setup_results_dir(project="ChaoSSM", phase="phase2b"):
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        base = f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base = os.path.abspath(f"./{project}_results/{phase}")
        print(f"[info] Not in Colab ({type(e).__name__}); using local dir.")
    os.makedirs(base, exist_ok=True); print("results ->", base); return base

RESULTS = setup_results_dir()
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=300,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)
def savejson(o,n):
    p=os.path.join(RESULTS,n)
    with open(p,"w") as fh: json.dump(o,fh,indent=2,default=str)
    print("saved",p)

torch 2.11.0+cu128 | device: cuda
Mounted at /content/drive
results -> /content/drive/My Drive/ChaoSSM/results/phase2b


## Configuration

In [2]:
SMOKE_TEST = False   # full run: 3 arms x 2 tasks x 6 seeds, then 3 robustness tests each

@dataclass
class Config:
    state_dim: int = 48
    d_model: int = 48
    vocab: int = 16
    steps: int = 1200
    batch: int = 64
    lr: float = 3e-4
    bptt_k: int = 64
    grad_clip: float = 1.0
    gamma_init: float = 1.0
    warmup_frac: float = 0.10
    dual_lr: float = 0.05
    alpha_max: float = 50.0
    corridor_target: float = 0.15
    corridor_half_width: float = 0.10
    lyap_window: int = 4
    lyap_subsample: int = 3
    lyap_every: int = 20
    copy_len: int = 12
    recall_pairs: int = 6
    arms: tuple = ("linear", "chaossm_free", "chaossm_corridor")
    tasks: tuple = ("copy", "assoc_recall")
    seeds: int = 6
    eval_batch: int = 256
    # robustness sweeps
    noise_stds: tuple = (0.0, 0.02, 0.05, 0.1, 0.2)
    quant_bits: tuple = (16, 8, 6, 4)
    finetune_steps: int = 50
    finetune_lr_mult: float = 10.0

CFG = Config()
if SMOKE_TEST:
    CFG.steps = 150; CFG.seeds = 2; CFG.eval_batch = 64; CFG.finetune_steps = 10
    print("*** SMOKE TEST — pipeline check only ***")
print(json.dumps({k: str(v) for k, v in asdict(CFG).items()}, indent=2))

{
  "state_dim": "48",
  "d_model": "48",
  "vocab": "16",
  "steps": "1200",
  "batch": "64",
  "lr": "0.0003",
  "bptt_k": "64",
  "grad_clip": "1.0",
  "gamma_init": "1.0",
  "warmup_frac": "0.1",
  "dual_lr": "0.05",
  "alpha_max": "50.0",
  "corridor_target": "0.15",
  "corridor_half_width": "0.1",
  "lyap_window": "4",
  "lyap_subsample": "3",
  "lyap_every": "20",
  "copy_len": "12",
  "recall_pairs": "6",
  "arms": "('linear', 'chaossm_free', 'chaossm_corridor')",
  "tasks": "('copy', 'assoc_recall')",
  "seeds": "6",
  "eval_batch": "256",
  "noise_stds": "(0.0, 0.02, 0.05, 0.1, 0.2)",
  "quant_bits": "(16, 8, 6, 4)",
  "finetune_steps": "50",
  "finetune_lr_mult": "10.0"
}


## 1. Model — unchanged from Phase 1/2a

In [3]:
def triangle_fold(x):
    return 1.0 - torch.abs(torch.remainder(x, 2.0) - 1.0)

class LinearMap(nn.Module):
    def __init__(self, N):
        super().__init__()
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_lin = nn.Parameter(W * 0.5)
    def forward(self, h):
        return torch.tanh(h @ self.W_lin.T)

class ChaoticMap(nn.Module):
    def __init__(self, N, gamma_init=1.0):
        super().__init__()
        self.w_r = nn.Parameter(torch.zeros(N))
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_mix = nn.Parameter(W * 0.1)
        inv = math.log(math.expm1(gamma_init))
        self.w_gamma = nn.Parameter(torch.tensor(float(inv)))
    def get_r(self): return 4.0 * torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def forward(self, h):
        r = self.get_r()
        u = h @ self.W_mix.T + r * h * (1 - h)
        return triangle_fold(self.get_gamma() * u)

class ChaoSSMCell(nn.Module):
    def __init__(self, cfg, phi_kind):
        super().__init__()
        self.N = cfg.state_dim
        self.phi_kind = phi_kind
        self.phi = LinearMap(cfg.state_dim) if phi_kind == "linear" \
                   else ChaoticMap(cfg.state_dim, cfg.gamma_init)
        self.A = nn.Parameter(torch.diag(torch.empty(cfg.state_dim).uniform_(0.9, 1.1)))
        self.B = nn.Parameter(torch.randn(cfg.state_dim, cfg.d_model) / math.sqrt(cfg.d_model))
        self.W_g = nn.Parameter(torch.randn(cfg.state_dim, cfg.d_model) / math.sqrt(cfg.d_model))
        self.b_g = nn.Parameter(torch.zeros(cfg.state_dim))
    def step(self, h, x):
        g = torch.sigmoid(x @ self.W_g.T + self.b_g)
        raw = g * self.phi(h @ self.A.T) + (1 - g) * h + x @ self.B.T
        return triangle_fold(raw) if self.phi_kind != "linear" else torch.tanh(raw)
    def init_state(self, b, dev):
        dtype = self.A.dtype   # match the model's actual parameter dtype (fixes fp16 inference)
        return torch.zeros(b, self.N, device=dev, dtype=dtype) if self.phi_kind == "linear" \
               else torch.full((b, self.N), 0.5, device=dev, dtype=dtype)

class ChaoSSMLM(nn.Module):
    def __init__(self, cfg, phi_kind, vocab_size, readout_size):
        super().__init__()
        self.cfg = cfg
        self.emb = nn.Embedding(vocab_size, cfg.d_model)
        self.cell = ChaoSSMCell(cfg, phi_kind)
        self.readout = nn.Linear(cfg.state_dim, readout_size)
    def forward(self, tok, return_traj=False, noise_std=0.0):
        B, T = tok.shape
        x = self.emb(tok)
        if noise_std > 0:
            x = x + torch.randn_like(x) * noise_std
        h = self.cell.init_state(B, tok.device)
        outs, traj = [], []
        for t in range(T):
            h = self.cell.step(h, x[:, t]); outs.append(self.readout(h))
            if return_traj: traj.append(h)
            if (t + 1) % self.cfg.bptt_k == 0: h = h.detach()
        lg = torch.stack(outs, 1)
        return (lg, torch.stack(traj, 1), x) if return_traj else lg

## 2. Estimator — unchanged

In [4]:
def _full_jac_fn(model, x):
    return lambda z: model.cell.step(z.unsqueeze(0), x.unsqueeze(0))[0]

def lambda_window(model, traj, xseq, W, n_sample):
    B, T, N = traj.shape
    if T <= W + 1:
        return torch.zeros((), device=traj.device)
    lams = []
    for b in torch.randperm(B)[:n_sample]:
        t0 = int(torch.randint(0, T - W - 1, (1,)))
        M = torch.eye(N, device=traj.device, dtype=traj.dtype)
        for k in range(W):
            M = jacrev(_full_jac_fn(model, xseq[b, t0+k]))(traj[b, t0+k]) @ M
        lams.append(torch.log(torch.linalg.matrix_norm(M, ord=2) + 1e-12) / W)
    return torch.stack(lams).mean()

## 3. Tasks — copy and associative recall, unchanged from Phase 1/2a

In [5]:
def make_copy_batch(cfg, batch, device, gen):
    L, V = cfg.copy_len, cfg.vocab
    sep, pad = V, V + 1
    src = torch.randint(0, V, (batch, L), device=device, generator=gen)
    inp = torch.cat([src, torch.full((batch,1), sep, device=device),
                     torch.full((batch,L), pad, device=device)], 1)
    tgt = torch.cat([torch.full((batch,L+1), -100, device=device), src], 1)
    return inp, tgt

def make_assoc_recall_batch(cfg, batch, device, gen):
    M, V = cfg.recall_pairs, cfg.vocab
    sep = V
    keys = torch.stack([torch.randperm(V, generator=gen, device=device)[:M] for _ in range(batch)])
    vals = torch.randint(0, V, (batch, M), device=device, generator=gen)
    qidx = torch.randint(0, M, (batch,), device=device, generator=gen)
    bidx = torch.arange(batch, device=device)
    qkey = keys[bidx, qidx]
    qval = vals[bidx, qidx]
    pairs = torch.stack([keys, vals], dim=2).reshape(batch, 2*M)
    inp = torch.cat([pairs, torch.full((batch,1), sep, device=device), qkey.unsqueeze(1)], 1)
    tgt = torch.full((batch, inp.shape[1]), -100, device=device)
    tgt[:, -1] = qval
    return inp, tgt

TASK_FNS = {"copy": make_copy_batch, "assoc_recall": make_assoc_recall_batch}
TASK_VOCAB = {"copy": CFG.vocab + 2, "assoc_recall": CFG.vocab + 1}
TASK_CHANCE = {"copy": 1.0 / CFG.vocab, "assoc_recall": 1.0 / CFG.vocab}

def task_accuracy(task, logits, tgt):
    if task == "copy":
        mask = tgt != -100
        pred = logits.argmax(-1)
        return ((pred == tgt) & mask).float().sum().item() / mask.float().sum().clamp(min=1).item()
    else:
        return (logits[:, -1].argmax(-1) == tgt[:, -1]).float().mean().item()

## 4. Trainer — unchanged mechanics

In [6]:
def train_one(arm, task, seed, cfg):
    torch.manual_seed(seed); np.random.seed(seed)
    gen = torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed)
    phi_kind = "linear" if arm == "linear" else "chaossm"
    vocab_size = TASK_VOCAB[task]
    model = ChaoSSMLM(cfg, phi_kind, vocab_size, vocab_size).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, betas=(0.9, 0.95))
    batch_fn = TASK_FNS[task]

    use_lyap = (arm == "chaossm_corridor")
    lo = hi = None
    if use_lyap:
        lo = cfg.corridor_target - cfg.corridor_half_width
        hi = cfg.corridor_target + cfg.corridor_half_width
    a_hi = a_lo = 0.0
    warmup_steps = int(cfg.warmup_frac * cfg.steps)
    diverged = False
    final_acc = float("nan")

    for step in range(cfg.steps):
        inp, tgt = batch_fn(cfg, cfg.batch, DEVICE, gen)
        need_traj = (arm != "linear")
        out = model(inp, return_traj=need_traj)
        logits, traj, xseq = out if need_traj else (out, None, None)
        L_task = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), tgt.reshape(-1),
                                 ignore_index=-100)

        L_pen = torch.zeros((), device=DEVICE); lam_val = float("nan")
        if arm != "linear":
            in_warmup = step < warmup_steps
            lam = lambda_window(model, traj, xseq, cfg.lyap_window, cfg.lyap_subsample)
            lam_val = float(lam.detach())
            if use_lyap and not in_warmup:
                L_pen = a_hi * F.relu(lam - hi) + a_lo * F.relu(lo - lam)

        opt.zero_grad(set_to_none=True); (L_task + L_pen).backward()
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip).item()
        opt.step()

        if use_lyap and step >= warmup_steps:
            a_hi = float(np.clip(a_hi + cfg.dual_lr * (lam_val - hi), 0.0, cfg.alpha_max))
            a_lo = float(np.clip(a_lo + cfg.dual_lr * (lo - lam_val), 0.0, cfg.alpha_max))

        if not (math.isfinite(L_task.item()) and math.isfinite(gn)):
            diverged = True; break

        if step % 20 == 0:
            with torch.no_grad():
                final_acc = task_accuracy(task, logits, tgt)

    return model, dict(arm=arm, task=task, seed=seed, diverged=diverged,
                       final_acc=final_acc, lam_settled=lam_val)

## 5. Train the base models

In [7]:
models = {}
base_rows = []
total = len(CFG.arms) * len(CFG.tasks) * CFG.seeds
done = 0
for task in CFG.tasks:
    for arm in CFG.arms:
        for sd in range(CFG.seeds):
            model, s = train_one(arm, task, sd, CFG)
            models[(arm, task, sd)] = model
            base_rows.append(s); done += 1
            flag = " DIVERGED" if s["diverged"] else ""
            print(f"[{done}/{total}] {task:14s} {arm:18s} seed={sd}  "
                  f"final_acc={s['final_acc']:.4f} (chance {TASK_CHANCE[task]:.4f})  "
                  f"lam={s['lam_settled']:+.3f}{flag}")

BASE = pd.DataFrame(base_rows)
savetable(BASE, "phase2b_base_summary.csv")

[1/36] copy           linear             seed=0  final_acc=0.0742 (chance 0.0625)  lam=+nan
[2/36] copy           linear             seed=1  final_acc=0.0573 (chance 0.0625)  lam=+nan
[3/36] copy           linear             seed=2  final_acc=0.0885 (chance 0.0625)  lam=+nan
[4/36] copy           linear             seed=3  final_acc=0.0469 (chance 0.0625)  lam=+nan
[5/36] copy           linear             seed=4  final_acc=0.0521 (chance 0.0625)  lam=+nan
[6/36] copy           linear             seed=5  final_acc=0.0573 (chance 0.0625)  lam=+nan
[7/36] copy           chaossm_free       seed=0  final_acc=0.1536 (chance 0.0625)  lam=+0.331
[8/36] copy           chaossm_free       seed=1  final_acc=0.1523 (chance 0.0625)  lam=+0.603
[9/36] copy           chaossm_free       seed=2  final_acc=0.1289 (chance 0.0625)  lam=+0.354
[10/36] copy           chaossm_free       seed=3  final_acc=0.1667 (chance 0.0625)  lam=+0.215
[11/36] copy           chaossm_free       seed=4  final_acc=0.0820 (cha

## 6. Test 1 — input perturbation robustness

Gaussian noise added to the embedded input at increasing standard deviation; accuracy measured
at each level on a fresh, frozen-model forward pass.

In [8]:
@torch.no_grad()
def eval_perturbation(model, task, cfg, seed_offset=20_000):
    gen = torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed_offset)
    inp, tgt = TASK_FNS[task](cfg, cfg.eval_batch, DEVICE, gen)
    accs = {}
    for std in cfg.noise_stds:
        logits = model(inp, noise_std=std)
        accs[std] = task_accuracy(task, logits, tgt)
    return accs

pert_rows = []
for (arm, task, sd), model in models.items():
    accs = eval_perturbation(model, task, CFG)
    row = dict(arm=arm, task=task, seed=sd, **{f"acc_std_{k}": v for k, v in accs.items()})
    row["auc_drop"] = accs[CFG.noise_stds[0]] - np.mean(list(accs.values())[1:])  # clean minus mean-perturbed
    pert_rows.append(row)

PERT = pd.DataFrame(pert_rows)
savetable(PERT, "phase2b_perturbation.csv")
print(PERT.groupby(["task","arm"]).auc_drop.agg(["mean","std"]).round(4).to_string())

saved /content/drive/My Drive/ChaoSSM/results/phase2b/phase2b_perturbation.csv
                                 mean     std
task         arm                             
assoc_recall chaossm_corridor  0.0166  0.0135
             chaossm_free      0.0140  0.0147
             linear            0.0034  0.0027
copy         chaossm_corridor  0.0622  0.0119
             chaossm_free      0.0473  0.0190
             linear           -0.0005  0.0024


## 7. Test 2 — quantization sensitivity

Symmetric uniform quantization applied to every weight tensor at decreasing bit-width, plus a
direct fp16-vs-fp32 comparison. All evaluated on a fresh forward pass against a temporary
quantized copy of the model — the original trained weights are never modified.

In [9]:
def quantize_tensor(t, bits):
    if bits >= 32: return t
    qmax = 2 ** (bits - 1) - 1
    scale = t.abs().max() / qmax if t.abs().max() > 0 else 1.0
    return torch.clamp(torch.round(t / scale), -qmax, qmax) * scale

@torch.no_grad()
def eval_quantization(model, task, cfg, seed_offset=20_000):
    gen = torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed_offset)
    inp, tgt = TASK_FNS[task](cfg, cfg.eval_batch, DEVICE, gen)
    results = {}
    for bits in cfg.quant_bits:
        qmodel = pycopy.deepcopy(model)
        with torch.no_grad():
            for p in qmodel.parameters():
                p.copy_(quantize_tensor(p.data, bits))
        logits = qmodel(inp)
        results[f"bits_{bits}"] = task_accuracy(task, logits, tgt)
    # fp16 inference (native, not simulated)
    try:
        model_fp16 = pycopy.deepcopy(model).half()
        logits16 = model_fp16(inp.clone(), noise_std=0.0)
        results["fp16"] = task_accuracy(task, logits16.float(), tgt)
    except Exception as e:
        results["fp16"] = float("nan")
        if not eval_quantization._warned:
            print(f"[warning] fp16 inference failed ({type(e).__name__}: {e}); "
                  f"fp16 column will be NaN for all subsequent evaluations. "
                  f"If this is a dtype-mismatch error, it indicates a real bug (a state or "
                  f"buffer not following the model's .half() cast) rather than a CPU/GPU "
                  f"limitation -- do not assume it will resolve itself on the GPU run.")
            eval_quantization._warned = True
    return results
eval_quantization._warned = False

quant_rows = []
for (arm, task, sd), model in models.items():
    r = eval_quantization(model, task, CFG)
    row = dict(arm=arm, task=task, seed=sd, **r)
    quant_rows.append(row)

QUANT = pd.DataFrame(quant_rows)
savetable(QUANT, "phase2b_quantization.csv")
lowest_bits_col = f"bits_{min(CFG.quant_bits)}"
print(QUANT.groupby(["task","arm"])[[f"bits_{b}" for b in CFG.quant_bits] + ["fp16"]].mean().round(4).to_string())

saved /content/drive/My Drive/ChaoSSM/results/phase2b/phase2b_quantization.csv
                               bits_16  bits_8  bits_6  bits_4    fp16
task         arm                                                      
assoc_recall chaossm_corridor   0.0885  0.0931  0.0788  0.0605  0.0879
             chaossm_free       0.0918  0.0885  0.0788  0.0723  0.0905
             linear             0.1758  0.1738  0.1686  0.1458  0.1758
copy         chaossm_corridor   0.1628  0.1578  0.1137  0.0621  0.1626
             chaossm_free       0.1401  0.1383  0.1049  0.0656  0.1407
             linear             0.0660  0.0653  0.0702  0.0636  0.0660


## 8. Test 3 — fine-tuning stability

A short burst (`finetune_steps`) of further training at `finetune_lr_mult`x the original
learning rate on fresh batches of the same task — an aggressive, deliberately destabilizing
perturbation, in the spirit of Halloran et al.'s framing of fine-tuning robustness. Accuracy is
measured immediately before and after.

In [10]:
def eval_finetune_stability(model, arm, task, seed, cfg):
    model = pycopy.deepcopy(model)
    gen = torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed + 30_000)

    with torch.no_grad():
        inp0, tgt0 = TASK_FNS[task](cfg, cfg.eval_batch, DEVICE, gen)
        acc_before = task_accuracy(task, model(inp0), tgt0)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr * cfg.finetune_lr_mult, betas=(0.9, 0.95))
    for _ in range(cfg.finetune_steps):
        inp, tgt = TASK_FNS[task](cfg, cfg.batch, DEVICE, gen)
        logits = model(inp)
        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), tgt.reshape(-1), ignore_index=-100)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        opt.step()

    with torch.no_grad():
        inp1, tgt1 = TASK_FNS[task](cfg, cfg.eval_batch, DEVICE, gen)
        acc_after = task_accuracy(task, model(inp1), tgt1)

    return dict(arm=arm, task=task, seed=seed, acc_before=acc_before, acc_after=acc_after,
               drop=acc_before - acc_after)

ft_rows = []
for (arm, task, sd), model in models.items():
    r = eval_finetune_stability(model, arm, task, sd, CFG)
    ft_rows.append(r)
    print(f"{task:14s} {arm:18s} seed={sd}  before={r['acc_before']:.4f}  "
          f"after={r['acc_after']:.4f}  drop={r['drop']:+.4f}")

FT = pd.DataFrame(ft_rows)
savetable(FT, "phase2b_finetune.csv")

copy           linear             seed=0  before=0.0677  after=0.0563  drop=+0.0114
copy           linear             seed=1  before=0.0615  after=0.0654  drop=-0.0039
copy           linear             seed=2  before=0.0986  after=0.0807  drop=+0.0179
copy           linear             seed=3  before=0.0560  after=0.0638  drop=-0.0078
copy           linear             seed=4  before=0.0618  after=0.0605  drop=+0.0013
copy           linear             seed=5  before=0.0742  after=0.0563  drop=+0.0179
copy           chaossm_free       seed=0  before=0.1647  after=0.0654  drop=+0.0993
copy           chaossm_free       seed=1  before=0.1579  after=0.0648  drop=+0.0931
copy           chaossm_free       seed=2  before=0.1530  after=0.0622  drop=+0.0908
copy           chaossm_free       seed=3  before=0.1416  after=0.0570  drop=+0.0846
copy           chaossm_free       seed=4  before=0.0986  after=0.0693  drop=+0.0293
copy           chaossm_free       seed=5  before=0.1266  after=0.0658  drop=

## 9. Statistical comparison across arms, per task

In [11]:
def paired(df, task, a, b, col):
    x = df[(df.task==task)&(df.arm==a)].sort_values("seed")[col].values
    y = df[(df.task==task)&(df.arm==b)].sort_values("seed")[col].values
    m = np.isfinite(x) & np.isfinite(y); x, y = x[m], y[m]
    if len(x) < 3 or np.allclose(x, y):
        return dict(task=task, pair=f"{a} vs {b}", metric=col, n=len(x), p=np.nan,
                    median_diff=float(np.median(x-y)) if len(x) else np.nan)
    st, p = wilcoxon(x, y)
    return dict(task=task, pair=f"{a} vs {b}", metric=col, n=len(x), p=float(p),
               median_diff=float(np.median(x-y)))

rows = []
for task in CFG.tasks:
    rows.append(paired(PERT, task, "linear", "chaossm_free", "auc_drop"))
    rows.append(paired(QUANT, task, "linear", "chaossm_free", lowest_bits_col))
    rows.append(paired(FT, task, "linear", "chaossm_free", "drop"))
    rows.append(paired(FT, task, "chaossm_corridor", "chaossm_free", "drop"))

ROBUST_STATS = pd.DataFrame(rows)
print(ROBUST_STATS.to_string(index=False))
savetable(ROBUST_STATS, "phase2b_stats.csv")
print(f"\nmin achievable two-sided p at n={CFG.seeds}: {2/(2**CFG.seeds):.4f}")

        task                             pair   metric  n       p  median_diff
        copy           linear vs chaossm_free auc_drop  6 0.03125    -0.053467
        copy           linear vs chaossm_free   bits_4  6 0.56250    -0.002604
        copy           linear vs chaossm_free     drop  6 0.03125    -0.080404
        copy chaossm_corridor vs chaossm_free     drop  6 0.03125     0.020020
assoc_recall           linear vs chaossm_free auc_drop  6 0.18750    -0.014160
assoc_recall           linear vs chaossm_free   bits_4  6 0.03125     0.066406
assoc_recall           linear vs chaossm_free     drop  6 0.84375    -0.007812
assoc_recall chaossm_corridor vs chaossm_free     drop  6 1.00000     0.003906
saved /content/drive/My Drive/ChaoSSM/results/phase2b/phase2b_stats.csv

min achievable two-sided p at n=6: 0.0312


In [12]:
fig, ax = plt.subplots(2, 2, figsize=(13, 9))
colors = {"linear":"gray", "chaossm_free":"tab:blue", "chaossm_corridor":"tab:green"}

for i, task in enumerate(CFG.tasks):
    for arm in CFG.arms:
        d = PERT[(PERT.task==task)&(PERT.arm==arm)]
        means = [d[f"acc_std_{s}"].mean() for s in CFG.noise_stds]
        ax[0,i].plot(CFG.noise_stds, means, marker="o", label=arm, color=colors[arm])
    ax[0,i].set(xlabel="input noise std", ylabel="accuracy", title=f"Perturbation robustness — {task}")
    ax[0,i].legend(fontsize=8); ax[0,i].grid(alpha=.3)

for i, task in enumerate(CFG.tasks):
    for arm in CFG.arms:
        d = QUANT[(QUANT.task==task)&(QUANT.arm==arm)]
        means = [d[f"bits_{b}"].mean() for b in CFG.quant_bits]
        ax[1,i].plot(CFG.quant_bits, means, marker="o", label=arm, color=colors[arm])
    ax[1,i].set(xlabel="quantization bits", ylabel="accuracy", title=f"Quantization sensitivity — {task}")
    ax[1,i].invert_xaxis()
    ax[1,i].legend(fontsize=8); ax[1,i].grid(alpha=.3)

fig.tight_layout(); savefig(fig, "phase2b_perturbation_quantization.png"); plt.close(fig)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for i, task in enumerate(CFG.tasks):
    FT[FT.task==task].boxplot(column="drop", by="arm", ax=ax[i])
    ax[i].set(title=f"Fine-tuning-induced accuracy drop — {task}", xlabel="arm", ylabel="drop")
fig.suptitle(""); fig.tight_layout(); savefig(fig, "phase2b_finetune_stability.png"); plt.close(fig)

saved /content/drive/My Drive/ChaoSSM/results/phase2b/phase2b_perturbation_quantization.png
saved /content/drive/My Drive/ChaoSSM/results/phase2b/phase2b_finetune_stability.png


## 10. Verdict

| Observation | Reading |
|---|---|
| chaossm arms show significantly larger perturbation/quantization/fine-tuning drops than linear on both tasks | Direct confirmation of Halloran et al.'s framing — the λ≤0 stability that linear SSMs get "for free" is a real, now-quantified cost of ChaoSSM's design, on top of the task-level trade-offs already found. Report as a genuine limitation alongside the task wins. |
| Robustness costs appear mainly on associative recall, not copy | Consistent with the Phase 2a finding that recall's representational structure is fragile to chaos in general — the robustness cost may be the same underlying fragility showing up under a different kind of perturbation, not a separate phenomenon. |
| chaossm_corridor is more robust than chaossm_free | The corridor's plateau-holding regularization has a stability benefit beyond the task-accuracy benefit already found — worth highlighting as an additional argument for the corridor mechanism's value. |
| No significant robustness gap anywhere | The training-time decay findings in §5.2/§5.4 were the dominant signal; post-hoc inference-time perturbation does not add a further measurable cost at this scale. A genuinely reassuring result, worth reporting as such rather than treated as a null finding to downplay. |

In [13]:
def flat_dict(series_groupby):
    # (task, arm) tuple index -> "task|arm" string keys, JSON-safe
    d = series_groupby.agg(["mean", "std"]).round(4)
    return {f"{k[0]}|{k[1]}": v for k, v in d.to_dict("index").items()}

verdict = dict(
    smoke_test=SMOKE_TEST,
    base_accuracy=flat_dict(BASE.groupby(["task","arm"]).final_acc),
    perturbation_auc_drop=flat_dict(PERT.groupby(["task","arm"]).auc_drop),
    quantization_lowest_bits=flat_dict(QUANT.groupby(["task","arm"])[lowest_bits_col]),
    finetune_drop=flat_dict(FT.groupby(["task","arm"]).drop),
    stats=ROBUST_STATS.to_dict("records"),
    config=dict(steps=CFG.steps, seeds=CFG.seeds, tasks=CFG.tasks,
                noise_stds=CFG.noise_stds, quant_bits=CFG.quant_bits,
                finetune_steps=CFG.finetune_steps, finetune_lr_mult=CFG.finetune_lr_mult),
)
savejson(verdict, "phase2b_verdict.json")
print(json.dumps(verdict, indent=2, default=str))

saved /content/drive/My Drive/ChaoSSM/results/phase2b/phase2b_verdict.json
{
  "smoke_test": false,
  "base_accuracy": {
    "assoc_recall|chaossm_corridor": {
      "mean": 0.1094,
      "std": 0.0356
    },
    "assoc_recall|chaossm_free": {
      "mean": 0.099,
      "std": 0.0274
    },
    "assoc_recall|linear": {
      "mean": 0.1641,
      "std": 0.0404
    },
    "copy|chaossm_corridor": {
      "mean": 0.1547,
      "std": 0.0207
    },
    "copy|chaossm_free": {
      "mean": 0.1317,
      "std": 0.0323
    },
    "copy|linear": {
      "mean": 0.0627,
      "std": 0.0156
    }
  },
  "perturbation_auc_drop": {
    "assoc_recall|chaossm_corridor": {
      "mean": 0.0166,
      "std": 0.0135
    },
    "assoc_recall|chaossm_free": {
      "mean": 0.014,
      "std": 0.0147
    },
    "assoc_recall|linear": {
      "mean": 0.0034,
      "std": 0.0027
    },
    "copy|chaossm_corridor": {
      "mean": 0.0622,
      "std": 0.0119
    },
    "copy|chaossm_free": {
      "mean": 0